In [223]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## **Load Data**

In [238]:
print(f"Current working directory: {os.getcwd()}")
os.chdir("/Users/derekwu/Desktop/Data/01_silver")
print(f"Current working directory: {os.getcwd()}")

data = pd.read_csv("clean_chs_waittimes.csv")

Current working directory: /Users/derekwu/Desktop/Data/01_silver
Current working directory: /Users/derekwu/Desktop/Data/01_silver


## **Encoding**

In [239]:
from sklearn.preprocessing import LabelEncoder

# Encode 'id' and 'clinic_type'
label_encoder_id = LabelEncoder()
label_encoder_clinic_type = LabelEncoder()

data['id_encoded'] = label_encoder_id.fit_transform(data['id'])
data['clinic_type_encoded'] = label_encoder_clinic_type.fit_transform(data['clinic_type'])

# Drop original categorical columns to avoid redundancy
data = data.drop(columns=['id', 'clinic_type'])

# Map days of the week to numeric values
day_mapping = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}

data['day_of_week_encoded'] = data['day_of_week'].map(day_mapping)

data = data.dropna()

In [240]:
for col in data.columns:
    print(col)

lastUpdated
queue
wait_time
treat_time
total_time
hour_minute
hour_minute_numeric
day_of_week
hour_time
queue_diff
wait_time_diff
treat_time_diff
total_time_diff
hour_minute_numeric_diff
id_encoded
clinic_type_encoded
day_of_week_encoded


## **Model**

In [241]:
set(data['clinic_type_encoded'])

{0, 1}

In [242]:
# Split data into training and testing sets
X = data[data['clinic_type_encoded'] == 0][['queue', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded']]
y = data[data['clinic_type_encoded'] == 0][['wait_time']]

# Align indices
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [243]:
def calculate_mape(y_true, y_pred):
    """
    Calculates Mean Absolute Percentage Error (MAPE).
    """
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_smape(y_true, y_pred):
    """
    Calculates Symmetric Mean Absolute Percentage Error (SMAPE).
    """
    return 100 * np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [258]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import itertools
import numpy as np

def evaluate_arima_with_optimization(y_train, y_test, p_range, d_range, q_range):
    """
    Optimizes ARIMA parameters and evaluates the best model.
    """
    best_order = None
    best_mse = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_smape = float('inf')
    best_model = None

    # Generate all combinations of p, d, q
    pdq_combinations = list(itertools.product(p_range, d_range, q_range))

    for order in pdq_combinations:
        try:
            # Fit ARIMA model
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()
            
            # Forecast on the test set
            y_pred = model_fit.forecast(steps=len(y_test))
            
            # Calculate Metrics
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mape = calculate_mape(y_test, y_pred)
            smape = calculate_smape(y_test, y_pred)

            # Update best model
            if mse < best_mse:
                best_mse, best_rmse, best_mape, best_smape = mse, rmse, mape, smape
                best_order = order
                best_model = model_fit
        except Exception as e:
            print(f"ARIMA fitting failed for order {order}: {e}")
            continue
    
    # Handle cases where no valid model was found
    if best_model is None:
        return None, None, float('nan'), float('nan'), float('nan'), float('nan')
    
    return best_model, best_order, best_mse, best_rmse, best_mape, best_smape

In [259]:
# Define the custom grouped time-series splitter
class GroupedTimeSeriesSplit:
    def __init__(self, group_col, time_col, n_splits):
        self.group_col = group_col
        self.time_col = time_col
        self.n_splits = n_splits

    def split(self, X, y=None):
        """Yield train-test splits for each group."""
        groups = X[self.group_col].unique()
        for group in groups:
            group_data = X[X[self.group_col] == group].sort_values(self.time_col)
            n_samples = len(group_data)
            
            # Skip small groups
            if n_samples <= self.n_splits:
                print(f"Skipping group '{group}' due to insufficient samples.")
                continue
            
            fold_size = n_samples // (self.n_splits + 1)
            if fold_size == 0:
                print(f"Skipping group '{group}' due to insufficient samples per fold.")
                continue

            for i in range(1, self.n_splits + 1):
                start_train = 0
                end_train = fold_size * i
                start_test = fold_size * i
                end_test = fold_size * (i + 1)

                # Ensure indices are within bounds
                if end_train > n_samples or end_test > n_samples:
                    print(f"Skipping fold {i} for group '{group}' due to index out-of-bounds.")
                    continue

                train_idx = group_data.index[start_train:end_train]
                test_idx = group_data.index[start_test:end_test]

                # Ensure indices are not empty
                if len(train_idx) == 0 or len(test_idx) == 0:
                    print(f"Skipping fold {i} for group '{group}' due to empty train/test indices.")
                    continue

                yield train_idx, test_idx

# Define a function for model evaluation
def evaluate_model_with_grouped_cv(model, X, y, splitter, p_range, d_range, q_range):
    mse_scores, rmse_scores, mape_scores, smape_scores, r2_scores = [], [], [], [], []
    
    for train_idx, test_idx in splitter.split(X):
        try:
            # Extract train-test splits
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx].values.ravel(), y.iloc[test_idx].values.ravel()
            
            # Skip if training data is too small
            if len(y_train) < 10:
                print("Skipping split due to insufficient training data.")
                continue
            
            if model == 'ARIMA':
                # Evaluate ARIMA model
                best_model, best_order, mse, rmse, mape, smape = evaluate_arima_with_optimization(
                    y_train, y_test, p_range, d_range, q_range
                )
            else:
                # Evaluate other models
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                
                # # Debug prediction shapes
                # print("y_pred shape:", y_pred.shape)
                # print("y_test shape:", y_test.shape)
                
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mape = calculate_mape(y_test, y_pred)
                smape = calculate_smape(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

            # Append metrics
            mse_scores.append(mse)
            rmse_scores.append(rmse)
            mape_scores.append(mape)
            smape_scores.append(smape)
            r2_scores.append(r2)

        except Exception as e:
            print(f"Error evaluating model: {e}")
            continue
    
    # Return average metrics across folds
    return (
        np.nanmean(mse_scores),
        np.nanmean(rmse_scores),
        np.nanmean(mape_scores),
        np.nanmean(smape_scores),
        np.nanmean(r2_scores),
    )

In [260]:
# Define ARIMA parameter ranges
p_range = range(0, 3)  # AR terms
d_range = range(0, 2)  # Differencing terms
q_range = range(0, 3)  # MA terms

# Define models
models = pd.DataFrame({
    'model_name': ['Linear Regression', 'Random Forest', 'XGBoost', 'ARIMA'],
    'model': [
        LinearRegression(),
        RandomForestRegressor(random_state=42),
        XGBRegressor(random_state=42),
        'ARIMA'  # Placeholder for ARIMA
    ]
})

# Instantiate the custom splitter
custom_splitter = GroupedTimeSeriesSplit(
    group_col='clinic_type_encoded',
    time_col='hour_minute_numeric',
    n_splits=4
)

In [261]:
# Apply the evaluation function
models[['mse', 'rmse', 'mape', 'smape', 'r2']] = models['model'].apply(
    lambda m: pd.Series(evaluate_model_with_grouped_cv(m, X, y, custom_splitter, p_range, d_range, q_range))
)

# Display results
print(models)

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimizat

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA param

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value
Error evaluating model: cannot access local variable 'r2' where it is not associated with a value
          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   
3              ARIMA                                              ARIMA   

        mse      rmse       mape      smape        r2  
0  0.575085  0.725132  27.066989  28.871060  0.179879  
1  0.272133  0.511360  18.945863  18.723752  0.573992  
2  0.291656  0.528698  19.527965  19.098442  0.545024  
3  0.719937  0.834627  30.641941  29.907580       NaN  


/var/folders/my/1zd7hfx53qq246r8xfrrzkdc0000gn/T/ipykernel_42134/1780320840.py:98: RuntimeWarning: Mean of empty slice
  np.nanmean(r2_scores),
